DATA PROCESSING + MODEL TRAINING

In [1]:
import os
import random
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import CLIPProcessor, CLIPModel
import os
from huggingface_hub import login

def safe_login():
    token = os.getenv("HF_TOKEN")

    if not token:
        try:
            with open("Hugging_Face_Token.txt") as f:
                token = f.read().strip()
        except FileNotFoundError:
            pass

    if token:
        login(token)
    else:
        print("Running without Hugging Face login.")

safe_login()

# =========================
# CONFIG
# =========================
DATA_DIR = "Data/img"
BATCH_SIZE = 32
EPOCHS = 20
K = 4
LR = 5e-6

CHECKPOINT_PATH = "checkpoint.pth"
BEST_MODEL_PATH = "best_model.pth"
SPLIT_PATH = "splits.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# STEP 1: BUILD DATAFRAME
# =========================
data = []

for root, _, files in os.walk(DATA_DIR):
    for file in files:
        if file.endswith(".jpg"):
            path = os.path.join(root, file)

            item_id = None
            for p in path.split(os.sep):
                if p.startswith("id_"):
                    item_id = p
                    break

            if item_id:
                data.append([path, item_id])

df = pd.DataFrame(data, columns=["image_path", "item_id"])
print("Total images:", len(df))

# =========================
# STEP 2: CORRECT SPLIT (KEY FIX)
# =========================
if os.path.exists(SPLIT_PATH):
    print("Loading existing splits...")
    df = pd.read_csv(SPLIT_PATH)
else:
    print("Creating retrieval-friendly splits...")

    df["split"] = "train"

    for item_id in df["item_id"].unique():
        idxs = df[df["item_id"] == item_id].index.tolist()

        if len(idxs) < 2:
            continue

        np.random.shuffle(idxs)

        split_point = int(0.8 * len(idxs))

        gallery_idxs = idxs[:split_point]
        query_idxs = idxs[split_point:]

        df.loc[gallery_idxs, "split"] = "gallery"
        df.loc[query_idxs, "split"] = "query"

    df.to_csv(SPLIT_PATH, index=False)
    print("Splits saved!")

print(df["split"].value_counts())

# IMPORTANT: train uses ALL data (not just train split)
train_df = df.copy().reset_index(drop=True)

# =========================
# DATASET
# =========================
class FashionDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

        self.label_map = {k: i for i, k in enumerate(df["item_id"].unique())}
        self.df["label"] = self.df["item_id"].map(self.label_map)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")

        inputs = self.processor(images=image, return_tensors="pt")

        return {
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "label": torch.tensor(row["label"])
        }

train_dataset = FashionDataset(train_df)

# =========================
# PK SAMPLER
# =========================
id_to_indices = defaultdict(list)

for idx in range(len(train_dataset)):
    item_id = train_dataset.df.iloc[idx]["item_id"]
    id_to_indices[item_id].append(idx)

class PKSampler(Sampler):
    def __init__(self, id_to_indices, batch_size, k):
        self.id_to_indices = id_to_indices
        self.item_ids = list(id_to_indices.keys())
        self.batch_size = batch_size
        self.k = k

    def __iter__(self):
        batch = []
        random.shuffle(self.item_ids)

        for item_id in self.item_ids:
            indices = self.id_to_indices[item_id]

            if len(indices) < self.k:
                continue

            sampled = random.sample(indices, self.k)
            batch.extend(sampled)

            if len(batch) >= self.batch_size:
                yield from batch[:self.batch_size]
                batch = []

    def __len__(self):
        return len(self.item_ids)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=PKSampler(id_to_indices, BATCH_SIZE, K),
    num_workers=4
)

# =========================
# MODEL
# =========================
model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32",
    use_safetensors=True
).to(device)

# Freeze text encoder
for p in model.text_model.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(model.vision_model.parameters(), lr=LR)
scaler = torch.cuda.amp.GradScaler()

# =========================
# LOAD CHECKPOINT
# =========================
start_epoch = 0
best_loss = float("inf")

if os.path.exists(CHECKPOINT_PATH):
    print("Loading checkpoint...")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)

    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_epoch = ckpt["epoch"] + 1
    best_loss = ckpt["best_loss"]

    print(f"Resuming from epoch {start_epoch}")

# =========================
# LOSS
# =========================
def contrastive_loss(features, labels, temp=0.07):
    features = F.normalize(features, dim=1)
    sim = torch.matmul(features, features.T)

    labels = labels.unsqueeze(1)
    mask = torch.eq(labels, labels.T).float().to(device)

    logits = sim / temp
    logits_mask = torch.ones_like(mask) - torch.eye(mask.size(0)).to(device)

    mask *= logits_mask
    exp_logits = torch.exp(logits) * logits_mask

    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-8)
    mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-8)

    return -mean_log_prob_pos.mean()

# =========================
# TRAIN
# =========================
for epoch in range(start_epoch, EPOCHS):
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for batch in loop:
        x = batch["pixel_values"].to(device)
        y = batch["label"].to(device)

        with torch.cuda.amp.autocast():
            feats = model.vision_model(pixel_values=x).pooler_output
            loss = contrastive_loss(feats, y)

        optimizer.zero_grad()
        scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} Avg Loss: {avg_loss:.4f}")

    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "best_loss": best_loss
    }, CHECKPOINT_PATH)

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("Saved best model!")

print("Training complete!")

Total images: 52712
Loading existing splits...
split
gallery    40054
query      12646
train         12
Name: count, dtype: int64


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_625017/3040915560.py:187: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Loading checkpoint...
Resuming from epoch 25
Training complete!


MODEL EVALUATION AND INFERENCE CODE

In [ ]:
import os
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
import os
from huggingface_hub import login

def safe_login():
    token = os.getenv("HF_TOKEN")

    if not token:
        try:
            with open("Hugging_Face_Token.txt") as f:
                token = f.read().strip()
        except FileNotFoundError:
            pass

    if token:
        login(token)
    else:
        print("Running without Hugging Face login.")

safe_login()
# =========================
# CONFIG
# =========================
DATA_DIR = "Data/img"
MODEL_PATH = "best_model.pth"
SPLIT_PATH = "splits.csv"
BATCH_SIZE = 64

device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# LOAD SPLITS (CRITICAL FIX)
# =========================
df = pd.read_csv(SPLIT_PATH)

# remove any hidden whitespace
df["item_id"] = df["item_id"].astype(str).str.strip()

query_df = df[df["split"] == "query"].reset_index(drop=True)
gallery_df = df[df["split"] == "gallery"].reset_index(drop=True)

print("Query size:", len(query_df))
print("Gallery size:", len(gallery_df))

# =========================
# DATASET
# =========================
class FashionDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")

        inputs = self.processor(images=image, return_tensors="pt")

        return {
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "item_id": str(row["item_id"]).strip()
        }

query_loader = DataLoader(FashionDataset(query_df), batch_size=BATCH_SIZE)
gallery_loader = DataLoader(FashionDataset(gallery_df), batch_size=BATCH_SIZE)

# =========================
# LOAD MODEL
# =========================
model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32",
    use_safetensors=True
).to(device)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()

# =========================
# EMBEDDINGS
# =========================
def extract_embeddings(loader):
    embeddings, ids = [], []

    with torch.no_grad():
        for batch in tqdm(loader):
            x = batch["pixel_values"].to(device)

            feats = model.vision_model(pixel_values=x).pooler_output
            feats = F.normalize(feats, dim=1)

            embeddings.append(feats.cpu())
            ids.extend([str(i).strip() for i in batch["item_id"]])

    return torch.cat(embeddings), ids

print("\nExtracting gallery embeddings...")
gallery_emb, gallery_ids = extract_embeddings(gallery_loader)

print("\nExtracting query embeddings...")
query_emb, query_ids = extract_embeddings(query_loader)

# =========================
# DEBUG CHECK (VERY IMPORTANT)
# =========================
query_set = set(query_ids)
gallery_set = set(gallery_ids)

print("\nDEBUG INFO:")
print("Unique query IDs:", len(query_set))
print("Unique gallery IDs:", len(gallery_set))
print("Overlap:", len(query_set & gallery_set))

if len(query_set & gallery_set) == 0:
    print("❌ ERROR: No overlap → metrics will be zero!")
    exit()

# =========================
# METRICS
# =========================
def compute_metrics(query_emb, gallery_emb, query_ids, gallery_ids, K_values=[5,10,15]):
    results = {}

    for K in K_values:
        recall_list, ap_list, ndcg_list = [], [], []

        for i in range(len(query_emb)):
            sims = torch.matmul(query_emb[i], gallery_emb.T)
            topk = torch.topk(sims, K).indices

            retrieved_ids = [gallery_ids[j] for j in topk]
            rel = np.array([1 if rid == query_ids[i] else 0 for rid in retrieved_ids])

            # Recall@K
            recall_list.append(1 if rel.sum() > 0 else 0)

            # mAP@K
            precisions = []
            correct = 0
            for idx, r in enumerate(rel):
                if r:
                    correct += 1
                    precisions.append(correct / (idx + 1))
            ap_list.append(np.mean(precisions) if precisions else 0)

            # NDCG@K
            dcg = sum(rel[j] / np.log2(j+2) for j in range(len(rel)))
            ideal = sorted(rel, reverse=True)
            idcg = sum(ideal[j] / np.log2(j+2) for j in range(len(rel)))
            ndcg_list.append(dcg / idcg if idcg > 0 else 0)

        results[K] = {
            "Recall": np.mean(recall_list),
            "mAP": np.mean(ap_list),
            "NDCG": np.mean(ndcg_list)
        }

    return results

# =========================
# RUN EVALUATION
# =========================
metrics = compute_metrics(query_emb, gallery_emb, query_ids, gallery_ids)

print("\n===== FINAL METRICS =====")
for K, vals in metrics.items():
    print(f"\n@{K}")
    print(f"Recall@{K}: {vals['Recall']:.4f}")
    print(f"mAP@{K}:    {vals['mAP']:.4f}")
    print(f"NDCG@{K}:   {vals['NDCG']:.4f}")

Query size: 12646
Gallery size: 40054


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Extracting gallery embeddings...


  0%|          | 0/626 [00:00<?, ?it/s]


Extracting query embeddings...


  0%|          | 0/198 [00:00<?, ?it/s]

FINAL VISUALISATION

In [ ]:
import os
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel

# =========================
# CONFIG
# =========================
MODEL_PATH = "best_model.pth"
SPLIT_PATH = "splits.csv"
BATCH_SIZE = 64
TOP_K = 5

device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(SPLIT_PATH)
df["item_id"] = df["item_id"].astype(str).str.strip()

query_df = df[df["split"] == "query"].reset_index(drop=True)
gallery_df = df[df["split"] == "gallery"].reset_index(drop=True)

print("Query:", len(query_df), "| Gallery:", len(gallery_df))

# =========================
# DATASET
# =========================
class FashionDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")

        inputs = self.processor(images=image, return_tensors="pt")

        return {
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "item_id": str(row["item_id"]).strip(),
            "path": row["image_path"]
        }

query_loader = DataLoader(FashionDataset(query_df), batch_size=BATCH_SIZE)
gallery_loader = DataLoader(FashionDataset(gallery_df), batch_size=BATCH_SIZE)

# =========================
# LOAD MODEL
# =========================
print("Loading model...")
model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32",
    use_safetensors=True
).to(device)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()

# =========================
# EXTRACT EMBEDDINGS
# =========================
def extract_embeddings(loader):
    embeddings, ids, paths = [], [], []

    with torch.no_grad():
        for batch in tqdm(loader):
            x = batch["pixel_values"].to(device)

            feats = model.vision_model(pixel_values=x).pooler_output
            feats = F.normalize(feats, dim=1)

            embeddings.append(feats.cpu())
            ids.extend(batch["item_id"])
            paths.extend(batch["path"])

    return torch.cat(embeddings), ids, paths

print("Extracting gallery embeddings...")
gallery_emb, gallery_ids, gallery_paths = extract_embeddings(gallery_loader)

print("Extracting query embeddings...")
query_emb, query_ids, query_paths = extract_embeddings(query_loader)

# =========================
# VISUALIZATION FUNCTION
# =========================
def visualize(query_idx):
    q_vec = query_emb[query_idx]
    q_id = query_ids[query_idx]

    sims = torch.matmul(q_vec, gallery_emb.T)
    topk_vals, topk_idx = torch.topk(sims, TOP_K)

    plt.figure(figsize=(15, 4))

    # Query image
    plt.subplot(1, TOP_K + 1, 1)
    img = Image.open(query_paths[query_idx])
    plt.imshow(img)
    plt.title("QUERY", fontsize=12, color="blue")
    plt.axis("off")

    # Retrieved images
    for i, idx in enumerate(topk_idx):
        plt.subplot(1, TOP_K + 1, i + 2)

        img = Image.open(gallery_paths[idx])
        plt.imshow(img)

        # Check correctness
        correct = gallery_ids[idx] == q_id
        color = "green" if correct else "red"

        score = topk_vals[i].item()

        plt.title(
            f"Top {i+1}\n{score:.2f}",
            color=color,
            fontsize=10
        )

        # Border effect
        for spine in plt.gca().spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)

        plt.axis("off")

    plt.tight_layout()
    plt.show()

# =========================
# RUN DEMO
# =========================
print("\nShowing sample results...\n")

for i in range(5):
    visualize(i)